# 保存した学習を試す

一覧から学習 token を選び、指標と残差を確認して、新しい入力を予測します。上から順番にセルを実行してください。
この例は保存ファイルを読み取ります。再学習やライブのテーブル読取は行いません。

In [ ]:
from IPython.display import HTML, JSON, SVG, display
from tools.local_experiment.mlflow.training_runs import collect
from tools.local_experiment.notebook.training import load_saved_review, render_review, residual_svg
from tools.local_experiment.mlflow.predict import predict

## 1. 学習の一覧

`runs` にある `training_token` と `kind` を確認します。完了時刻の順番はモデル品質の順位ではありません。新しく学習した後は、このセルから実行し直してください。

In [ ]:
saved_runs = collect()
display(JSON(saved_runs))

## 2. 使う学習を選ぶ

次の空文字列に、一覧の token を貼り付けて実行します。保存した指標・holdout・残差が表示されます。

In [ ]:
training_token = ""
chosen = next((run for run in saved_runs["runs"] if run["training_token"] == training_token), None)
review = None
if chosen is None:
    print("一覧にある training_token を入力してください。")
else:
    review = load_saved_review(training_token, kind=chosen["kind"])
    display(HTML(render_review(review)))
    display(SVG(residual_svg(review)))

## 3. 入力を変更して予測する

Iceberg の学習を選んだ場合に実行できます。`x1`・`x2`・`x3` を編集し、セルを再実行してください。1〜64 行の有限数値に対応します。これは保存モデルの計算結果であり、新しい入力での正解精度を保証するものではありません。

Notebook を保存すると、入力と表示結果もこのローカル workspace に残ります。

In [ ]:
prediction_rows = [
    {"x1": 0.0, "x2": 0.0, "x3": 0.0},
    {"x1": 0.25, "x2": -0.75, "x3": 1.5},
]
if review is None:
    print("先に学習を選択してレビューのセルを実行してください。")
elif chosen["kind"] != "iceberg":
    print("予測には kind が iceberg の学習を選択してください。")
else:
    prediction = predict(training_token, {"schema_version": 1, "rows": prediction_rows})
    display(JSON(prediction))